# `kitai.batch` — Guide

This notebook walks through every function in `kitai/batch.py` in the order
you would call them in a real workflow.

## Module overview

```
kitai/batch.py
│
├── Generic Batch API primitives          (any endpoint)
│   ├── submit_batch_job()                task list → job_id
│   ├── check_batch_job()                 job_id → status dict
│   ├── download_batch_results()          job_id → raw result list
│   └── BatchJobNotCompleteError          raised when downloading too early
│
├── Embedding workflow helpers            (/v1/embeddings)
│   ├── build_embedding_tasks()           Documents → task dicts
│   ├── poll_until_complete()             blocks until all jobs finish
│   └── parse_embedding_results()         raw results → (custom_id, embedding) pairs
│
└── Chat completion workflow helpers      (/v1/chat/completions)
    ├── build_chat_tasks()                items + system_prompt → task dicts
    └── parse_chat_results()              raw results + extractor_fn → (custom_id, T) pairs
```

## Sections
1. [Setup](#1-setup)
2. [Prepare documents](#2-prepare-documents)
3. [Embedding pipeline — step by step](#3-embedding-pipeline)
4. [Chat completion pipeline](#4-chat-completion-pipeline)
5. [Generic Batch API — custom task dicts](#5-generic-batch-api)
6. [Error handling reference](#6-error-handling)

---
## 1 — Setup

Imports, logging, and the OpenAI client.

> **Prerequisite:** `OPENAI_API_KEY` must be set in your `.env` file or
> environment before running the cells below.

In [ ]:
import logging
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from langchain_core.documents import Document

from kitai.batch import (
    submit_batch_job,
    check_batch_job,
    download_batch_results,
    poll_until_complete,
    build_embedding_tasks,
    parse_embedding_results,
    build_chat_tasks,
    parse_chat_results,
    BatchJobNotCompleteError,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_EMBEDDING_DIMENSIONS,
    DEFAULT_CHAT_MODEL,
)

# Route kitai.batch logs to the notebook output.
# Switch to DEBUG to see every upload ID and poll tick.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)

load_dotenv()
client = OpenAI()

print(f"Default embedding model     : {DEFAULT_EMBEDDING_MODEL}")
print(f"Default embedding dimensions: {DEFAULT_EMBEDDING_DIMENSIONS}")
print(f"Default chat model          : {DEFAULT_CHAT_MODEL}")

---
## 2 — Prepare documents

`build_embedding_tasks` requires a list of LangChain `Document` objects where
every document carries a unique `metadata["id"]`.

`kitai.transform` provides helpers for building that list from different sources:

| Function | Responsibility |
|---|---|
| `list_to_docs(texts)` | `list[str]` → `list[Document]` |
| `df_to_docs(df, content_column, metadata_columns)` | `DataFrame` → `list[Document]` |
| `add_metadata_to_docs(docs, key, value)` | attach a constant metadata field to every doc |

After conversion, assign a unique integer `metadata["id"]` to each document —
this becomes the `custom_id` key in the Batch API tasks and lets you join
results back to your source data.

In [ ]:
from langchain_core.documents import Document
from kitai.transform import list_to_docs, df_to_docs

# --- Option A: build from a list of strings (quick demo) -----------------
raw_texts = [
    "Algorithmic trading involves using computer programs to execute trades.",
    "Short selling is a strategy where an investor borrows shares to sell them.",
    "Risk management is essential for any long/short equity strategy.",
]
docs = list_to_docs(raw_texts)

# Assign a unique metadata["id"] to each document (required by the batch pipeline)
for i, doc in enumerate(docs, start=1):
    doc.metadata["id"] = i

# --- Option B: build from a DataFrame ------------------------------------
# import pandas as pd
# df = pd.read_csv("./data/my_data.csv")
# docs = df_to_docs(df, content_column="description", metadata_columns=["source"])
# for i, doc in enumerate(docs, start=1):
#     doc.metadata["id"] = i

print(f"Documents loaded: {len(docs)}")
for doc in docs:
    print(f"  id={doc.metadata['id']}  text={doc.page_content[:60]}...")

---
## 3 — Embedding pipeline

The full flow in five steps:

```
docs
 └─► build_embedding_tasks()   — format for the Batch API
      └─► submit_batch_job()   — upload + create job → job_id
           └─► poll_until_complete()  — wait
                └─► download_batch_results()  — pull raw JSONL
                     └─► parse_embedding_results()  → [(custom_id, vector), ...]
```

### Step 1 — Build embedding tasks

`build_embedding_tasks` converts your documents into the dict schema the
OpenAI Batch API expects.  `custom_id` is set to `"custom_id_{doc.metadata['id']}"`
so you can join the results back to your original data later.

In [ ]:
tasks = build_embedding_tasks(
    docs,
    model=DEFAULT_EMBEDDING_MODEL,
    dimensions=DEFAULT_EMBEDDING_DIMENSIONS,
)

print(f"Tasks built: {len(tasks)}")
print("\nFirst task:")
tasks[0]

### Step 2 — Submit the batch job

`submit_batch_job` serialises the task list to a temporary JSONL file,
uploads it to the OpenAI Files API, creates the batch job, and returns
the job ID.  The temp file is deleted regardless of success or failure.

The `metadata` dict is optional but helps you identify the job later in
the OpenAI dashboard.

In [ ]:
job_id = submit_batch_job(
    client,
    tasks,
    endpoint="/v1/embeddings",
    metadata={"description": "batch_api_guide_demo"},
)

print(f"Job ID: {job_id}")
# Save this — you can paste it into a later cell if the kernel restarts.
# job_id = "batch_abc123"

### Step 3 — Check status (manual spot-check)

Use `check_batch_job` to inspect the current state of a single job
without blocking.  Useful for debugging or when you want to poll manually.

The returned dict always contains:

| Key | Type | Meaning |
|---|---|---|
| `status` | str | e.g. `"in_progress"`, `"completed"` |
| `is_terminal` | bool | will the status still change? |
| `is_complete` | bool | `True` only when `status == "completed"` |
| `counts` | dict | `{total, completed, failed}` |
| `output_file_id` | str\|None | set when complete |
| `error_file_id` | str\|None | set when there are failed items |

In [ ]:
status = check_batch_job(client, job_id)

print(f"Status      : {status['status']}")
print(f"Is complete : {status['is_complete']}")
print(f"Is terminal : {status['is_terminal']}")
print(f"Counts      : {status['counts']}")
print(f"Output file : {status['output_file_id']}")

### Step 4 — Wait for completion

`poll_until_complete` blocks and polls every `poll_interval` seconds
until all jobs reach a terminal state.  It returns only the job IDs
that ended with `"completed"` — failed/expired/cancelled jobs are logged
at ERROR but not raised, so you can handle partial success.

**Tips:**
- Use `poll_interval=1.0` for short jobs in development.
- For production batches use `30.0`–`60.0` to avoid hammering the API.
- You can pass multiple job IDs if you submitted more than one batch.

In [ ]:
completed_ids = poll_until_complete(
    client,
    batch_ids=[job_id],
    poll_interval=5.0,   # seconds between polls
)

print(f"Completed jobs : {completed_ids}")

if not completed_ids:
    raise RuntimeError("All jobs failed — check the logs above for details.")

### Step 5 — Download raw results

`download_batch_results` calls `check_batch_job` internally and raises
`BatchJobNotCompleteError` if the job is not yet done.  Once complete
it downloads the output JSONL and returns one dict per submitted task.

Each raw result dict looks like:
```json
{
    "custom_id": "custom_id_1",
    "response": {
        "status_code": 200,
        "body": { "data": [ { "embedding": [...] } ] }
    },
    "error": null
}
```
Items that failed at the API level will have a non-null `"error"` field.

In [ ]:
results = download_batch_results(client, completed_ids[0])

print(f"Result objects : {len(results)}")
print(f"Failed items   : {sum(1 for r in results if r.get('error'))}")
print("\nFirst raw result (truncated):")
first = results[0].copy()
first["response"]["body"]["data"][0]["embedding"] = [
    *first["response"]["body"]["data"][0]["embedding"][:4], "..."
]
first

### Step 6 — Parse embeddings

`parse_embedding_results` extracts `(custom_id, embedding)` pairs from the
raw result list.  Items with errors are skipped and logged — the caller
receives only the successfully parsed pairs.

In [ ]:
pairs = parse_embedding_results(results)

print(f"Parsed pairs : {len(pairs)}")
for custom_id, embedding in pairs:
    print(f"  {custom_id}  dim={len(embedding)}  first_values={embedding[:3]}")

### Step 7 — Save to CSV (optional)

Convert pairs to a DataFrame and persist.  The `id` column is parsed
from the `custom_id` string so it can be joined back to your source data.

In [ ]:
df_embeddings = pd.DataFrame(pairs, columns=["custom_id", "embedding"])
df_embeddings["id"] = (
    df_embeddings["custom_id"]
    .str.replace("custom_id_", "", regex=False)
    .astype(int)
)
df_embeddings = df_embeddings[["id", "custom_id", "embedding"]]

output_path = "./book_embeddings.csv"
df_embeddings.to_csv(output_path, index=False)

print(f"Saved {len(df_embeddings)} rows to {output_path}")
df_embeddings.head()

---
## 4 — Chat completion pipeline

`build_chat_tasks` and `parse_chat_results` mirror the embedding helpers for
`/v1/chat/completions`.

Each item in the input list must have:

| Key | Purpose |
|---|---|
| `"id"` | Becomes `custom_id` — used to match results back to inputs. |
| `"content"` | The user-turn message sent to the model. |

The system prompt is shared across all tasks in the batch.
`parse_chat_results` accepts an `extractor_fn` that maps the raw response
string to any output type you need (plain string, parsed JSON, a Pydantic
model, etc.).

In [ ]:
questions = [
    "What is the core idea behind short selling?",
    "What does 'alpha' mean in quantitative finance?",
    "Define 'drawdown' in one sentence.",
]

# Each item needs "id" and "content"; system prompt is shared across all tasks.
items = [{"id": str(i), "content": q} for i, q in enumerate(questions)]

chat_tasks = build_chat_tasks(
    items,
    system_prompt="Answer in one concise sentence.",
    model=DEFAULT_CHAT_MODEL,
)

print(f"Built {len(chat_tasks)} chat tasks")
chat_tasks[0]

In [ ]:
chat_job_id = submit_batch_job(
    client,
    chat_tasks,
    endpoint="/v1/chat/completions",
    metadata={"description": "finance_qa_demo"},
)
print(f"Chat batch job: {chat_job_id}")

In [ ]:
chat_done = poll_until_complete(client, [chat_job_id], poll_interval=5.0)
chat_results = download_batch_results(client, chat_done[0])

# parse_chat_results applies extractor_fn to each choices[0].message.content.
# Item-level errors are logged at ERROR and skipped — not raised.
pairs = parse_chat_results(chat_results, extractor_fn=str.strip)

print(f"Parsed {len(pairs)} result(s)\n")
for custom_id, answer in pairs:
    print(f"[{custom_id}] {answer[:120]}")

---
## 5 — Generic Batch API

The three primitives (`submit_batch_job`, `check_batch_job`,
`download_batch_results`) are endpoint-agnostic.  Use them when you need
full control over task structure — e.g. setting `max_tokens`, injecting
per-item model overrides, or hitting a future OpenAI Batch endpoint.

Build task dicts manually and pass them straight to `submit_batch_job`
with the matching `endpoint` argument.  Parse the raw results yourself
by inspecting `item["response"]["body"]`.

In [ ]:
# Simulate calling download before a job is done.
# Replace with a real in-progress job ID to test against live API.
fake_in_progress_id = "batch_notdoneyetXXX"

try:
    download_batch_results(client, fake_in_progress_id)
except BatchJobNotCompleteError as e:
    print(f"Caught BatchJobNotCompleteError")
    print(f"  .batch_id = {e.batch_id}")
    print(f"  .status   = {e.status}")
    if e.status in ("in_progress", "finalizing", "validating"):
        print("  → Job is still running. Call poll_until_complete() to wait.")
    else:
        print(f"  → Terminal failure ({e.status}). Check the OpenAI dashboard.")
except Exception as e:
    # Will hit openai.NotFoundError for a fake ID — expected in this demo.
    print(f"API error (expected for fake ID): {type(e).__name__}: {e}")

---
## 6 — Error handling reference

### `BatchJobNotCompleteError`

Raised by `download_batch_results` when the job is not yet `"completed"`.
Check `.status` to decide whether to retry or abort.

```
status               meaning                     action
─────────────────    ─────────────────────────   ─────────────────────────
"validating"         OpenAI checking the file    poll again later
"in_progress"        running                     poll again later
"finalizing"         writing output file         poll again later
"failed"             hard failure                inspect error_file_id
"expired"            exceeded completion_window  resubmit
"cancelled"          user-cancelled              resubmit if needed
```

### `ValueError` — empty inputs

All functions that take a list guard against empty input and raise
`ValueError` immediately, before any network call is made.

In [ ]:
guards = [
    ("build_embedding_tasks([])",       lambda: build_embedding_tasks([])),
    ("submit_batch_job(client, [])",    lambda: submit_batch_job(client, [])),
    ("poll_until_complete(client, [])", lambda: poll_until_complete(client, [])),
    ("parse_embedding_results([])",     lambda: parse_embedding_results([])),
    ("build_chat_tasks([], ...)",       lambda: build_chat_tasks([], system_prompt="s")),
    ("parse_chat_results([], ...)",     lambda: parse_chat_results([], extractor_fn=str.strip)),
]

for label, fn in guards:
    try:
        fn()
    except ValueError as e:
        print(f"{label}\n  → ValueError: {e}\n")

### Debugging tips

| Symptom | Where to look |
|---|---|
| Job never leaves `"validating"` | OpenAI is validating the uploaded file; wait a minute then re-check. |
| `status == "failed"` with `error_file_id` | Download the error file: `client.files.content(status['error_file_id']).text` |
| Embeddings have wrong dimension | Check the `dimensions` arg in `build_embedding_tasks` matches what your vector store expects. |
| `parse_embedding_results` skips items | Set `logging.DEBUG` to see the exact `custom_id` and error for every skipped item. |
| `parse_chat_results` skips items | Set `logging.DEBUG` — each skip logs `custom_id` + reason (batch error, bad shape, or `extractor_fn` exception). |
| Need to re-run download after kernel restart | Copy the job ID from the log output and set `job_id = "batch_xxx"` manually. |